# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [64]:
data = pd.read_csv("./data/AviationData.csv", encoding="Windows-1252", low_memory=False)
# Inspect NaNs:
# data.isna() # Multiple exist

# Inspect Datatypes:
# data.info() #~89K Rows + 31 Columns (mostly objects, some float64)

# Summary Statistics
data.describe()
# Number of Engines: mean=1.15, std=0.4
# Total Fatal Injuries: mean=0.65, std=5.49
# Total Serious Injuries: mean=0.28, std=1.55


,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [65]:
# INSPECT RELEVANT COLUMNS:
# data.columns
# data["Report.Status"].value_counts() # I changed this often to check the unfamiliar columns
relevant_columns = ['Aircraft.damage', 'Make', 'Model', 'Total.Fatal.Injuries', 
                    'Total.Serious.Injuries', 'Weather.Condition', 'Broad.phase.of.flight', 'Publication.Date']

# CREATE NEW DF
extracted_data = data[relevant_columns].copy()

# FILTER DATASET
# Step 1: dropna on publication data
extracted_data.dropna(subset="Publication.Date", inplace=True)
# Step 2: filter to only keep rows with dates from 01-01-2023
extracted_data["Publication.Date"] = extracted_data["Publication.Date"].astype(str)
extracted_data["Publication.Date"] = pd.to_datetime(extracted_data["Publication.Date"], format="%d-%m-%Y")
new_data = extracted_data[extracted_data["Publication.Date"] >= "1983-01-01"].copy()
new_data.head(10)


,Aircraft.damage,Make,Model,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight,Publication.Date
1,Destroyed,Piper,PA24-180,4.0,0.0,UNK,Unknown,1996-09-19
2,Destroyed,Cessna,172M,3.0,NaN,IMC,Cruise,2007-02-26
3,Destroyed,Rockwell,112,2.0,0.0,IMC,Cruise,2000-09-12
5,Substantial,Mcdonnell Douglas,DC9,NaN,NaN,VMC,Climb,2017-09-19
6,Destroyed,Cessna,180,4.0,0.0,IMC,Unknown,2001-11-06
12,Destroyed,Bellanca,17-30A,0.0,0.0,IMC,Cruise,1983-01-02
13,Destroyed,Cessna,R172K,1.0,0.0,IMC,Takeoff,1983-01-02
14,Destroyed,Navion,A,1.0,0.0,IMC,Cruise,1983-01-02
15,Destroyed,Beech,19,2.0,0.0,IMC,Cruise,1983-01-02
16,Destroyed,Enstrom,280C,0.0,0.0,IMC,Taxi,1983-01-02


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [66]:
# Total.Fatal.Injuries & Total.Serious.Injuries still have NaN's present
# Will drop rows where BOTH are NaN / using the pandas ~ operator as in a "NOT"
new_data = new_data[~(new_data["Total.Fatal.Injuries"].isna() & new_data["Total.Serious.Injuries"].isna())]
# Now I'll assume any remaining single NaN is a 0.0 value
new_data.loc[new_data["Total.Fatal.Injuries"].isna(), "Total.Fatal.Injuries"] = 0.0
new_data.loc[new_data["Total.Serious.Injuries"].isna(), "Total.Serious.Injuries"] = 0.0
new_data.head(10)


,Aircraft.damage,Make,Model,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight,Publication.Date
1,Destroyed,Piper,PA24-180,4.0,0.0,UNK,Unknown,1996-09-19
2,Destroyed,Cessna,172M,3.0,0.0,IMC,Cruise,2007-02-26
3,Destroyed,Rockwell,112,2.0,0.0,IMC,Cruise,2000-09-12
6,Destroyed,Cessna,180,4.0,0.0,IMC,Unknown,2001-11-06
12,Destroyed,Bellanca,17-30A,0.0,0.0,IMC,Cruise,1983-01-02
13,Destroyed,Cessna,R172K,1.0,0.0,IMC,Takeoff,1983-01-02
14,Destroyed,Navion,A,1.0,0.0,IMC,Cruise,1983-01-02
15,Destroyed,Beech,19,2.0,0.0,IMC,Cruise,1983-01-02
16,Destroyed,Enstrom,280C,0.0,0.0,IMC,Taxi,1983-01-02
17,Destroyed,Cessna,180,3.0,0.0,VMC,Unknown,1983-01-02


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [79]:
# View values
# new_data["Aircraft.damage"].value_counts()

# Convert NaN's to a string of "Unknown"
new_data.loc[new_data["Aircraft.damage"].isna(), "Aircraft.damage"] = "Unknown"

# Make new column with bool value of destroyed or not
new_data['is_destroyed'] = new_data['Aircraft.damage'] == 'Destroyed'
new_data.head(15)

,Aircraft.damage,Make,Model,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight,Publication.Date,is_destroyed
1,Destroyed,Piper,PA24-180,4.0,0.0,UNK,Unknown,1996-09-19,True
2,Destroyed,Cessna,172M,3.0,0.0,IMC,Cruise,2007-02-26,True
3,Destroyed,Rockwell,112,2.0,0.0,IMC,Cruise,2000-09-12,True
6,Destroyed,Cessna,180,4.0,0.0,IMC,Unknown,2001-11-06,True
12,Destroyed,Bellanca,17-30A,0.0,0.0,IMC,Cruise,1983-01-02,True
13,Destroyed,Cessna,R172K,1.0,0.0,IMC,Takeoff,1983-01-02,True
14,Destroyed,Navion,A,1.0,0.0,IMC,Cruise,1983-01-02,True
15,Destroyed,Beech,19,2.0,0.0,IMC,Cruise,1983-01-02,True
16,Destroyed,Enstrom,280C,0.0,0.0,IMC,Taxi,1983-01-02,True
17,Destroyed,Cessna,180,3.0,0.0,VMC,Unknown,1983-01-02,True


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [86]:
# View values
new_data["Make"].value_counts() # There are duplicates such as "Cessna" and "CESSNA"

# Normalize strings to uppercase and strip
new_data['Make'] = new_data['Make'].str.upper()
new_data['Make'] = new_data['Make'].str.strip()

# Remove makes where there are < 50
make_counts = new_data['Make'].value_counts()
new_data = new_data[new_data['Make'].isin(make_counts[make_counts >= 50].index)]

new_data.head(15)

,Aircraft.damage,Make,Model,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight,Publication.Date,is_destroyed
1,Destroyed,PIPER,PA24-180,4.0,0.0,UNK,Unknown,1996-09-19,True
2,Destroyed,CESSNA,172M,3.0,0.0,IMC,Cruise,2007-02-26,True
3,Destroyed,ROCKWELL,112,2.0,0.0,IMC,Cruise,2000-09-12,True
6,Destroyed,CESSNA,180,4.0,0.0,IMC,Unknown,2001-11-06,True
12,Destroyed,BELLANCA,17-30A,0.0,0.0,IMC,Cruise,1983-01-02,True
13,Destroyed,CESSNA,R172K,1.0,0.0,IMC,Takeoff,1983-01-02,True
14,Destroyed,NAVION,A,1.0,0.0,IMC,Cruise,1983-01-02,True
15,Destroyed,BEECH,19,2.0,0.0,IMC,Cruise,1983-01-02,True
16,Destroyed,ENSTROM,280C,0.0,0.0,IMC,Taxi,1983-01-02,True
17,Destroyed,CESSNA,180,3.0,0.0,VMC,Unknown,1983-01-02,True


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [ ]:
# View values
# new_data["Model"].value_counts() #Don't see anything crazy + NaN's are already removed


,Make,Model
1,PIPER,PA24-180
2,CESSNA,172M
3,ROCKWELL,112
6,CESSNA,180
12,BELLANCA,17-30A
...,...,...
88881,CESSNA,172F
88883,AIR TRACTOR,AT502
88884,PIPER,PA-28-151
88886,AMERICAN CHAMPION AIRCRAFT,8GCBC


### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [ ]:
# new_data["Weather.Condition"].value_counts() # Has duplicates: UNK & Unk
# Normalize strings to uppercase and strip
new_data['Weather.Condition'] = new_data['Weather.Condition'].str.upper()
new_data['Weather.Condition'] = new_data['Weather.Condition'].str.strip()

# new_data["Broad.phase.of.flight"].value_counts() # Has both unknown & other, will combine
new_data["Broad.phase.of.flight"] = new_data["Broad.phase.of.flight"].replace(['Unknown', 'Other'], 'other')   
new_data.head(15) #Looks good to me!


,Aircraft.damage,Make,Model,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight,Publication.Date,is_destroyed
1,Destroyed,PIPER,PA24-180,4.0,0.0,UNK,other,1996-09-19,True
2,Destroyed,CESSNA,172M,3.0,0.0,IMC,Cruise,2007-02-26,True
3,Destroyed,ROCKWELL,112,2.0,0.0,IMC,Cruise,2000-09-12,True
6,Destroyed,CESSNA,180,4.0,0.0,IMC,other,2001-11-06,True
12,Destroyed,BELLANCA,17-30A,0.0,0.0,IMC,Cruise,1983-01-02,True
13,Destroyed,CESSNA,R172K,1.0,0.0,IMC,Takeoff,1983-01-02,True
14,Destroyed,NAVION,A,1.0,0.0,IMC,Cruise,1983-01-02,True
15,Destroyed,BEECH,19,2.0,0.0,IMC,Cruise,1983-01-02,True
16,Destroyed,ENSTROM,280C,0.0,0.0,IMC,Taxi,1983-01-02,True
17,Destroyed,CESSNA,180,3.0,0.0,VMC,other,1983-01-02,True


### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [ ]:
# new_data.info()
new_data.isna().sum()
# Weather.Condition          2788
# Broad.phase.of.flight     19018
# However, I won't be removing these, becuase those columns are vital

Aircraft.damage               0
Make                          0
Model                        16
Total.Fatal.Injuries          0
Total.Serious.Injuries        0
Weather.Condition          2788
Broad.phase.of.flight     19018
Publication.Date              0
is_destroyed                  0
dtype: int64

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [103]:
new_data.to_csv('./data/cleaned_aviation_data.csv', index=False)

# Verify by reading the file just created
verify_data = pd.read_csv("./data/cleaned_aviation_data.csv")
verify_data

,Aircraft.damage,Make,Model,Total.Fatal.Injuries,Total.Serious.Injuries,Weather.Condition,Broad.phase.of.flight,Publication.Date,is_destroyed
0,Destroyed,PIPER,PA24-180,4.0,0.0,UNK,other,1996-09-19,True
1,Destroyed,CESSNA,172M,3.0,0.0,IMC,Cruise,2007-02-26,True
2,Destroyed,ROCKWELL,112,2.0,0.0,IMC,Cruise,2000-09-12,True
3,Destroyed,CESSNA,180,4.0,0.0,IMC,other,2001-11-06,True
4,Destroyed,BELLANCA,17-30A,0.0,0.0,IMC,Cruise,1983-01-02,True
...,...,...,...,...,...,...,...,...,...
53532,Unknown,CESSNA,172F,0.0,1.0,NaN,NaN,2022-12-22,False
53533,Unknown,AIR TRACTOR,AT502,1.0,0.0,NaN,NaN,2022-12-28,False
53534,Unknown,PIPER,PA-28-151,0.0,1.0,NaN,NaN,2022-12-29,False
53535,Substantial,AMERICAN CHAMPION AIRCRAFT,8GCBC,0.0,0.0,VMC,NaN,2022-12-27,False
